In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import pickle
import seaborn as sns
from bosperrus import *
plt.rcParams['svg.fonttype'] = 'none'


In [ ]:
pal = FIT_PALETTE

In [ ]:
cmap = "magma"

In [ ]:
np.random.seed(41)

In [ ]:
measure = "closeness"

In [ ]:
with open("../../mibitof_coords/coords.pickle", "rb") as f:
    datasets = pickle.load(f)

In [ ]:
dataset = "TNBC_mibitof:p14_labeledcellData.tiff"
coords = datasets[dataset]

In [ ]:
coords = coords[(coords[:, 0] < 1500) & (coords[:,1] < 1500)]
subset = coords /  (np.max(coords, axis=0) - np.min(coords, axis=0)) # [(coords[:, 0] < 1000) & (coords[:,1] < 1000)]

In [ ]:
G = nx.Graph()
G.add_nodes_from(range(len(subset)))
for n in G.nodes:
    G.nodes[n]["pos"] = subset[n]
edges = delaunay_edges(subset)
G.add_edges_from(edges)

In [ ]:
df =  pd.DataFrame(compute_centrality_measures(edges, len(subset), measures=[measure]))
df["distance"] = distance_to_convex_hull(subset)

In [ ]:
d = df["distance"]
C_true = df[measure].values   

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(5, 5))
pos = nx.get_node_attributes(G, "pos")
edges1 = nx.draw_networkx_edges(G, pos=pos, ax=ax, edge_color="black", width=0.5)
nodes1 = nx.draw_networkx_nodes(G, pos=pos, node_color=C_true, node_size=3, cmap=cmap, ax=ax)
edges1.set_rasterized(True)
nodes1.set_rasterized(True)
plt.axis("square")
plt.axis("off")
plt.savefig("fig1_closeness.svg")

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(5, 5))
edges1 = nx.draw_networkx_edges(G, pos=pos, ax=ax, edge_color="black", width=0.5)
nodes1 = nx.draw_networkx_nodes(G, pos=pos, node_color=d, node_size=3, cmap="viridis", ax=ax)
edges1.set_rasterized(True)
nodes1.set_rasterized(True)
plt.axis("square")
plt.axis("off")
plt.savefig("fig1_distance.svg")

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))

fitted = []
for fit_cls in (ConstantFit, PiecewiseLinearFit, MichaelisMentenFit, ExponentialSaturationFit, ExponentialDecayFit):
    f = fit_cls(S_true=C_true, d=d)
    f.fit()
    fitted.append(f)

d_grid = np.linspace(np.min(d), np.max(d), 200)
plot_fit(ax, d, C_true, fitted[0].predict, d_grid=d_grid, line_kwargs={"color": pal[fitted[0].name], "label": fitted[0].name})
for f in fitted[1:]:
    ax.plot(d_grid, f.predict(d_grid), color=pal[f.name], label=f.name, linewidth=2)

ax.set_xlabel("")
ax.set_ylabel("")
ax.axis("off")
plt.savefig("fig1_fits.svg")

In [ ]:
import matplotlib.colors as mcolors

f = Flow.from_distances_and_scores(distances=d, scores=df[["closeness"]])
f.flow()
best = f.best_fits["closeness"]
fit_color = pal[best.name]
plateau = float(best.predict(np.array([np.max(d)]))[0])
corrected = f.observations["BOSPERRUS corrected closeness"]

# corrected scores: the same quantile bands in light / mid magenta of the fit colour, mean in dark magenta
rgb = np.array(mcolors.to_rgb(fit_color))
light, mid, dark = (1 - 0.25) * np.ones(3) + 0.25 * rgb, (1 - 0.55) * np.ones(3) + 0.55 * rgb, 0.7 * rgb

fig, ax = plt.subplots(figsize=(5, 5))
plot_fit(ax, d, C_true, best.predict, line_kwargs={"color": fit_color, "linewidth": 2})
plot_fit(ax, d, corrected, lambda g: np.full_like(g, plateau),
         band_colors=(tuple(light) + (0.85,), tuple(mid) + (0.85,)),
         mean_kwargs={"color": tuple(dark), "linewidth": 1},
         line_kwargs={"color": fit_color, "linewidth": 1, "linestyle": "--"})
ax.axis("off")
plt.savefig("fig1_corrections.svg")

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(5, 5))
pos = nx.get_node_attributes(G, "pos")
edges1 = nx.draw_networkx_edges(G, pos=pos, ax=ax, edge_color="black", width=0.5)
nodes1 = nx.draw_networkx_nodes(G, pos=pos, node_color=f.observations["BOSPERRUS corrected closeness"], node_size=3, cmap=cmap, ax=ax)
edges1.set_rasterized(True)
nodes1.set_rasterized(True)
plt.axis("square")
plt.axis("off")

plt.savefig("fig1_corrected_closeness.svg")